### Objective

Convert the processed customer churn dataset into a SQLite database and answer business questions using SQL.

### Section 1 — Import Libraries

In [1]:
import sqlite3
import pandas as pd
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

### Objective

This notebook demonstrates SQL-based business analytics by loading the processed churn dataset into a SQLite database and executing business-oriented SQL queries.

### Section 2 — Load Processed Dataset

In [2]:
df = pd.read_csv("../data/processed/churn_processed.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (2850, 17)


,Call Failure,Complains,Subscription Length,Charge Amount,Seconds of Use,Frequency of use,Frequency of SMS,Distinct Called Numbers,Age Group,Tariff Plan,Status,Age,Customer Value,Churn,Customer_Engagement,Average_Call_Duration,Revenue_Per_Month
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0,93,60.694444,5.067692
1,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0,16,53.000000,1.150875
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0,443,40.213115,40.434737
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0,102,62.656716,6.154359
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0,93,40.559322,3.738590


### Section 3 — Create SQLite Database

In [3]:
os.makedirs("../sql", exist_ok=True)

conn = sqlite3.connect("../sql/churn.db")

cursor = conn.cursor()

print("Database created successfully.")

Database created successfully.


### Section 4 — Store Dataset into SQL

In [4]:
df.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

print("Dataset loaded into SQLite successfully.")

Dataset loaded into SQLite successfully.


### Section 5 — Verify Table

In [5]:
query = """
SELECT *
FROM customers
LIMIT 5;
"""

pd.read_sql(query, conn)

,Call Failure,Complains,Subscription Length,Charge Amount,Seconds of Use,Frequency of use,Frequency of SMS,Distinct Called Numbers,Age Group,Tariff Plan,Status,Age,Customer Value,Churn,Customer_Engagement,Average_Call_Duration,Revenue_Per_Month
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0,93,60.694444,5.067692
1,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0,16,53.000000,1.150875
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0,443,40.213115,40.434737
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0,102,62.656716,6.154359
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0,93,40.559322,3.738590


### Section 6 — Verify Columns

In [6]:
query = """
PRAGMA table_info(customers);
"""

pd.read_sql(query, conn)

,cid,name,type,notnull,dflt_value,pk
0,0,Call Failure,INTEGER,0,None,0
1,1,Complains,INTEGER,0,None,0
2,2,Subscription Length,INTEGER,0,None,0
3,3,Charge Amount,INTEGER,0,None,0
4,4,Seconds of Use,INTEGER,0,None,0
5,5,Frequency of use,INTEGER,0,None,0
6,6,Frequency of SMS,INTEGER,0,None,0
7,7,Distinct Called Numbers,INTEGER,0,None,0
8,8,Age Group,INTEGER,0,None,0
9,9,Tariff Plan,INTEGER,0,None,0


### Section 7 — Total Customers

In [7]:
query = """
SELECT COUNT(*) AS Total_Customers
FROM customers;
"""

pd.read_sql(query, conn)

,Total_Customers
0,2850


### Business Insight

The telecom company currently has **2850 customers** available for churn analysis.

### Section 8 — Total Churned Customers

In [8]:
query = """
SELECT
    COUNT(*) AS Churned_Customers
FROM customers
WHERE Churn = 1;
"""

pd.read_sql(query, conn)

,Churned_Customers
0,446


### Section 9 — Churn Rate

In [9]:
query = """
SELECT
ROUND(
100.0 * SUM(Churn) / COUNT(*),
2
) AS Churn_Rate
FROM customers;
"""

pd.read_sql(query, conn)

,Churn_Rate
0,15.65


### Business Insight

Approximately 15.65% of customers have churned.

This metric serves as the primary KPI for customer retention.

### Section 10 — Customer Status Analysis

Business Insight

Customers with Status 2 have a substantially higher churn rate than those with Status 1, indicating a need for targeted retention strategies.

### Section 11 — Complaint Analysis

In [10]:
query = """
SELECT
    Complains,
    COUNT(*) AS Total_Customers,
    SUM(Churn) AS Churned_Customers,
    ROUND(AVG(Churn)*100,2) AS Churn_Rate
FROM customers
GROUP BY Complains
ORDER BY Churn_Rate DESC;
"""

complaint_df = pd.read_sql(query, conn)
complaint_df

,Complains,Total_Customers,Churned_Customers,Churn_Rate
0,1,230,190,82.61
1,0,2620,256,9.77


### Business Insight

Customers who have registered complaints exhibit a significantly higher churn rate.

### Recommendation

Improve complaint resolution time and customer support quality to reduce churn.

### Section 12 — Tariff Plan Analysis

In [11]:
query = """
SELECT
    `Tariff Plan`,
    COUNT(*) AS Customers,
    ROUND(AVG(Churn)*100,2) AS Churn_Rate
FROM customers
GROUP BY `Tariff Plan`
ORDER BY Churn_Rate DESC;
"""

tariff_df = pd.read_sql(query, conn)
tariff_df

,Tariff Plan,Customers,Churn_Rate
0,1,2621,16.79
1,2,229,2.62


### Business Insight

Certain tariff plans have noticeably higher churn rates.

### Recommendation

Review pricing, benefits, and customer satisfaction for underperforming plans.

### Section 13 — Age Group Analysis

In [12]:
query = """
SELECT
    `Age Group`,
    COUNT(*) AS Customers,
    ROUND(AVG(Churn)*100,2) AS Churn_Rate
FROM customers
GROUP BY `Age Group`
ORDER BY Churn_Rate DESC;
"""

age_df = pd.read_sql(query, conn)
age_df

,Age Group,Customers,Churn_Rate
0,4,367,20.16
1,2,921,17.05
2,3,1296,16.44
3,5,154,1.30
4,1,112,0.00


### Business Insight

Some age groups are more likely to churn than others.

### Recommendation

Develop age-specific marketing campaigns and retention strategies.

### Section 14 — Customer Value Analysis

In [14]:
query = """
SELECT
    Churn,
    ROUND(AVG(`Customer Value`),2) AS Avg_Customer_Value
FROM customers
GROUP BY Churn;
"""

customer_value_df = pd.read_sql(query, conn)
customer_value_df

,Churn,Avg_Customer_Value
0,0,538.59
1,1,132.18


### Business Insight

Customers who remain with the company generally have higher customer value than those who churn.

### Recommendation

Identify and proactively retain high-value customers.

### Section 15 — Average Usage Analysis

In [15]:
query = """
SELECT
    Churn,
    ROUND(AVG(`Seconds of Use`),2) AS Avg_Seconds,
    ROUND(AVG(`Frequency of use`),2) AS Avg_Frequency,
    ROUND(AVG(`Frequency of SMS`),2) AS Avg_SMS
FROM customers
GROUP BY Churn;
"""

usage_df = pd.read_sql(query, conn)
usage_df

,Churn,Avg_Seconds,Avg_Frequency,Avg_SMS
0,0,5069.59,77.87,84.35
1,1,1648.66,30.70,16.85


### Business Insight

Customers with lower service usage are generally more likely to churn.

### Recommendation

Increase customer engagement through personalized offers and usage incentives.

### Section 16 — Top 10 High-Value Customers

In [16]:
query = """
SELECT
    `Customer Value`,
    Age,
    Status,
    Churn
FROM customers
ORDER BY `Customer Value` DESC
LIMIT 10;
"""

top_customers = pd.read_sql(query, conn)
top_customers

,Customer Value,Age,Status,Churn
0,2165.280,30,1,0
1,2149.280,30,1,0
2,2148.840,30,1,0
3,2148.030,25,1,0
4,2140.960,30,1,0
5,2129.535,25,1,0
6,2127.680,30,1,0
7,2124.840,30,1,0
8,2120.670,25,1,0
9,2117.720,30,1,0


### Business Insight

The highest-value customers represent a significant source of revenue and should receive personalized retention efforts.

### Section 17 — Ranking Customers (Window Function)

In [18]:
query = """
SELECT
    `Customer Value`,
    Churn,
    RANK() OVER(
        ORDER BY `Customer Value` DESC
    ) AS Customer_Rank
FROM customers
LIMIT 20;
"""

rank_df = pd.read_sql(query, conn)
rank_df

,Customer Value,Churn,Customer_Rank
0,2165.280,0,1
1,2149.280,0,2
2,2148.840,0,3
3,2148.030,0,4
4,2140.960,0,5
5,2129.535,0,6
6,2127.680,0,7
7,2124.840,0,8
8,2120.670,0,9
9,2117.720,0,10


### Business Insight

Window functions allow businesses to rank customers based on value without losing row-level detail.

### Section 18 — Common Table Expression (CTE)

In [19]:
query = """
WITH churn_summary AS
(
SELECT
    Status,
    COUNT(*) AS Customers,
    ROUND(AVG(Churn)*100,2) AS Churn_Rate
FROM customers
GROUP BY Status
)

SELECT *
FROM churn_summary
ORDER BY Churn_Rate DESC;
"""

cte_df = pd.read_sql(query, conn)
cte_df

,Status,Customers,Churn_Rate
0,2,684,47.51
1,1,2166,5.59


### Business Insight

Common Table Expressions simplify complex SQL logic and improve query readability.

### Section 19 — Export Results

In [20]:
import os

os.makedirs("../results/sql_results", exist_ok=True)

complaint_df.to_csv(
    "../results/sql_results/complaint_analysis.csv",
    index=False
)

tariff_df.to_csv(
    "../results/sql_results/tariff_analysis.csv",
    index=False
)

age_df.to_csv(
    "../results/sql_results/age_group_analysis.csv",
    index=False
)

customer_value_df.to_csv(
    "../results/sql_results/customer_value_analysis.csv",
    index=False
)

usage_df.to_csv(
    "../results/sql_results/service_usage_analysis.csv",
    index=False
)

rank_df.to_csv(
    "../results/sql_results/customer_ranking.csv",
    index=False
)

print("All SQL analysis reports exported successfully.")

All SQL analysis reports exported successfully.


### Section 20 — Executive Summary

### Executive Summary

#### Key Findings

- The dataset contains **2,850 customers**, with an overall churn rate of **15.65%**.
- Customers with complaints have a significantly higher churn rate.
- Customer Status is one of the strongest indicators of churn.
- Certain tariff plans experience higher churn than others.
- Lower service usage is associated with increased churn.
- High-value customers contribute disproportionately to business revenue and require focused retention efforts.

#### Business Recommendations

1. Strengthen customer support to reduce complaint-driven churn.
2. Implement targeted retention campaigns for high-risk customer segments.
3. Optimize tariff plans with high churn rates.
4. Increase customer engagement through personalized offers.
5. Monitor high-value customers proactively using predictive analytics.

#### Conclusion

SQL analytics complements the machine learning pipeline by providing interpretable, business-oriented insights. These findings enable data-driven decision-making and support proactive customer retention strategies.